# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. np.array([2, 4 ,6]) задает массив с элементами 2, 4 , 6   далее принт a/2 делит каждый элемент массива на 2, a // 4 делит каждый элемент на 4 без остатка и плавющей точки в итоге на экран выводится [1. 2. 3.] [0 1 1] 
2. создается матрица размерность 3 строки на 4 столбца, далее axis = 1 проходится по строкам слево направо а метод .sum() суммирует все так получается [4. 4. 4.]
3. создаются два массива первый массив одномерный второй такой же но метод reshape растягивает его до двумерного массива далее при попытке сложения двух массивов используется broadcasting так как два массива имеют разную размерность первый (3,) второй (3, 1) Numpy виртуально растягивает два массива вдоль недостоющей оси до общей формы (3, 3) получается что первый массив дублируется 3 раза по вертикали а второй 3 раза по горизонтали в итоге они наслаиваются друг на друга и получается
[[0 1 2]
 [1 2 3]
 [2 3 4]] 
4. создается массив x метод sqrt вычисляет квадратный корень каждого элемента затем суммирует все элементы после нахождения корня в итоге получится 6
5. массив в диапозоне от 1 до 5(5 не включительно) все элементы возводятся во вторую степень затем находится средне арифмитическое c помощью метода .mean() получается 7.5

In [17]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [18]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [19]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [20]:
# 1
left = clean.sum()
sep = amount[clean & (month == '2026-09')].sum()
oct1 = amount[clean & (month == '2026-10')].sum()


diff = ((oct1 - sep) / sep) * 100
print(f"Количество оставшихся строк: {left}")
print(f"Изменение с сентября по октября: {round(diff, 1)}%")
    

Количество оставшихся строк: 4739
Изменение с сентября по октября: -2.3%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [21]:
# 2

cats = np.unique(cat)
mon = np.unique(month)

table = np.zeros((len(cats), len(mon)))

for i, k in enumerate(cats):
    for j, m in enumerate(mon):
        table[i, j] = amount[clean & (cat == k) & (month == m)].sum()

rev_2 = table.sum(axis = 1)
best_ind = rev_2.argmax()

sep_cat = table[: , 0]
oct_cat =  table[: , 1]
oct_sep = (((oct_cat - sep_cat) / sep_cat) * 100)
worst_ind = oct_sep.argmin()
print(f" Таблица категория на месяц: {table}")     
print(f" Категория которая приносит больше всего выручки за два месяца: {cats[best_ind]} : {rev_2[best_ind]}")
print(f" Категория которая упала сильнее всего: {cats[worst_ind]}: {oct_sep[round(worst_ind, 1)]}%")

 Таблица категория на месяц: [[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
 Категория которая приносит больше всего выручки за два месяца: Электроника : 34370430.0
 Категория которая упала сильнее всего: Одежда: -7.214604400519248%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [45]:
# 3
table_1 = np.zeros((len(cats), len(mon)))

for i, k in enumerate(cats):
    for j, m in enumerate(mon):
        table_1[i, j] = clean[(cat == k) & (month == m)].sum()

mean_check = (table / table_1)

r = mean_check[:,1].round(1)

highest = r.argmax()
print(f""" Число заказов по категориям и месяцам : 
{table_1}""")

print(f"Средние чеки по категориям за октябрь{r}")
print(f"Самый высокий средний чек за октябрь: {cats[highest]} - {r[highest]}")


 Число заказов по категориям и месяцам : 
[[476. 476.]
 [265. 299.]
 [422. 435.]
 [678. 652.]
 [531. 505.]]
Средние чеки по категориям за октябрь[10768.5  3815.8  6158.6 13045.6 33600.2]
Самый высокий средний чек за октябрь: Электроника - 33600.2


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [23]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [24]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [25]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [26]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [27]:
# 8